# Retain-Only LoRA Fine-Tuning for Machine Unlearning

**Approach**: Continue fine-tuning the **existing** Factify LoRA adapter on **only** the retain set. The model naturally drifts away from forget-set knowledge via catastrophic forgetting, while reinforcing retain-set knowledge.

**References**:
- Maini et al. (2024) "TOFU: A Task of Fictitious Unlearning for LLMs" ([arXiv:2401.06121](https://arxiv.org/abs/2401.06121))
- Yao et al. (2023) "Large Language Model Unlearning" ([arXiv:2310.10683](https://arxiv.org/abs/2310.10683))

**Pipeline**:
1. Load base model + original Factify LoRA adapter (**keep adapter trainable, no merge**)
2. Continue fine-tuning the **same adapter** on retain set only
3. The adapter weights drift toward retain knowledge, forgetting the forget set
4. Merge final adapter → Evaluate FSR + RR per 5W category

In [ ]:
# Install dependencies
!pip install -q transformers peft accelerate safetensors bitsandbytes>=0.46.1 trl

In [ ]:
# HuggingFace login (needed for gated models)
from huggingface_hub import login
login()

In [ ]:
# ── Configuration ──────────────────────────────────────────

# Option A: Llama-3.2-3B
MODEL_CONFIG = {
    "base_model": "meta-llama/Llama-3.2-3B",
    "adapter": "Novaspree/factify-3B-adapter",
    "name": "Llama-3.2-3B",
}

# Option B: Gemma-3-4B (uncomment below, comment above)
# MODEL_CONFIG = {
#     "base_model": "google/gemma-3-4b-it",
#     "adapter": "Novaspree/factify-Gemma3-adapter-1",
#     "name": "Gemma-3-4B-IT",
# }

# ── Training hyperparameters ──────────────────────────────
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4  # effective batch = 16
MAX_SEQ_LENGTH = 256

# ── Dataset URLs ──────────────────────────────────────────
FORGET_SET_URL = "https://huggingface.co/datasets/Novaspree/factify_5K_enriched/resolve/main/forget/forget_set_fixed.json"
RETAIN_SET_URL = "https://huggingface.co/datasets/Novaspree/factify_5K_enriched/resolve/main/retain_set_fixed.json"

# ── Eval settings ─────────────────────────────────────────
MAX_NEW_TOKENS = 64

# ── Output ────────────────────────────────────────────────
OUTPUT_DIR = "./retain_only_model"

In [ ]:
# ── Load datasets ──────────────────────────────────────────
import json
import urllib.request
from collections import defaultdict

def load_dataset_from_url(url):
    print(f"Downloading: {url.split('/')[-1]}")
    with urllib.request.urlopen(url) as response:
        data = json.loads(response.read().decode())
    print(f"  Loaded {len(data)} samples")
    return data

forget_set = load_dataset_from_url(FORGET_SET_URL)
retain_set = load_dataset_from_url(RETAIN_SET_URL)

label_counts = defaultdict(int)
for item in retain_set:
    label_counts[item.get("label", "unknown")] += 1
print(f"\nRetain set by 5W category: {dict(label_counts)}")

## Step 1: Load base model + existing adapter (keep trainable)

We load the original Factify adapter but **do NOT merge it**. We keep it as a trainable LoRA so we can continue fine-tuning it on the retain set.

In [ ]:
# ── Load base model + existing adapter (trainable) ─────────
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Use 4-bit for Gemma, fp16 for Llama (fits on T4)
if "gemma" in MODEL_CONFIG["base_model"].lower():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    load_kwargs = {"quantization_config": bnb_config}
else:
    load_kwargs = {"torch_dtype": torch.float16}

print(f"Loading base model: {MODEL_CONFIG['base_model']}")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_CONFIG["base_model"],
    device_map="auto",
    trust_remote_code=True,
    **load_kwargs,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CONFIG["base_model"],
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load existing adapter — keep it trainable (is_trainable=True), NO merge
print(f"Loading adapter (trainable): {MODEL_CONFIG['adapter']}")
model = PeftModel.from_pretrained(base_model, MODEL_CONFIG["adapter"], is_trainable=True)
model.print_trainable_parameters()
print(f"✓ Model loaded with trainable Factify adapter. Will fine-tune on retain set.")

## Step 2: Prepare retain dataset for training

Format the retain set as causal LM training data. Same Q&A format as evaluation.

In [ ]:
# ── Prepare retain set for training ────────────────────────
from datasets import Dataset

def format_for_training(examples):
    """Format Q&A pairs as causal LM training text."""
    texts = []
    for q, a in zip(examples["question"], examples["answer"]):
        text = f"Question: {q}\nAnswer: {a}"
        texts.append(text)
    return {"text": texts}

retain_ds = Dataset.from_list(retain_set)
retain_ds = retain_ds.map(format_for_training, batched=True, remove_columns=retain_ds.column_names)

print(f"✓ Training dataset: {len(retain_ds)} samples")
print(f"  Example: {retain_ds[0]['text'][:120]}...")

## Step 3: Continue fine-tuning the existing adapter on retain set

We resume training the **same** Factify LoRA adapter, but now only on retain data. The adapter weights shift toward retain knowledge, and the forget-set facts are overwritten via catastrophic forgetting.

In [ ]:
# ── Fine-tune on retain set ────────────────────────────────
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    report_to="none",  # disable wandb
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=retain_ds,
    processing_class=tokenizer,
)

print("Starting retain-only fine-tuning...")
trainer.train()
print("✓ Training complete!")

In [ ]:
# ── Merge fine-tuned adapter into model for eval ───────────
model = model.merge_and_unload()
model.eval()
print(f"✓ Retain-fine-tuned adapter merged. Ready for evaluation.")

## Step 4: Evaluate FSR + RR per 5W category

In [ ]:
# ── Evaluation functions (same as adapter_negation notebook) ──
from tqdm.notebook import tqdm

@torch.no_grad()
def collect_predictions(model, tokenizer, dataset, split_name="forget"):
    """Run inference and collect question, answer, generation, and match."""
    model.eval()
    records = []

    for item in tqdm(dataset, desc=f"{split_name} set"):
        question = item["question"]
        target_answer = item["answer"].strip()
        category = item.get("label", "unknown")
        rephrases = item.get("rephrases", [])

        prompt = f"Question: {question}\nAnswer:"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

        generated = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        ).strip()

        is_match = target_answer.lower() in generated.lower()

        records.append({
            "question": question,
            "target_answer": target_answer,
            "model_generation": generated,
            "exact_match": is_match,
            "category": category,
            "split": split_name,
            "rephrases": rephrases,
            "model": MODEL_CONFIG["name"],
            "method": "retain_only_finetuning",
            "epochs": NUM_EPOCHS,
            "lr": LEARNING_RATE,
        })

    print(f"  ✓ {len(records)} predictions collected")
    return records


def compute_metrics(records, split_name):
    """Compute per-category and overall FSR/RR."""
    results_by_category = defaultdict(lambda: {"correct": 0, "total": 0})
    overall_correct = 0
    overall_total = 0

    for r in records:
        cat = r["category"]
        results_by_category[cat]["total"] += 1
        overall_total += 1
        if r["exact_match"]:
            results_by_category[cat]["correct"] += 1
            overall_correct += 1

    overall_rate = overall_correct / overall_total if overall_total > 0 else 0

    if split_name == "forget":
        metric_name = "FSR"
        print(f"\n{'='*50}")
        print(f"FORGET SET — FSR (model FAILS to recall, ↑ = better)")
    else:
        metric_name = "RR"
        print(f"\n{'='*50}")
        print(f"RETAIN SET — RR (model STILL recalls, ↑ = better)")

    print(f"{'='*50}")
    print(f"{'Category':<12} {'Samples':>8} {metric_name:>10}")
    print(f"{'-'*32}")

    for cat in sorted(results_by_category.keys()):
        r = results_by_category[cat]
        if split_name == "forget":
            val = 1.0 - (r["correct"] / r["total"])
        else:
            val = r["correct"] / r["total"]
        print(f"{cat:<12} {r['total']:>8} {val:>10.1%}")

    overall_val = (1.0 - overall_rate) if split_name == "forget" else overall_rate
    print(f"{'-'*32}")
    print(f"{'OVERALL':<12} {'':>8} {overall_val:>10.1%}")
    print(f"{'='*50}")

    return results_by_category, overall_val

In [ ]:
# ── Run evaluation ─────────────────────────────────────────
print(f"MODEL: {MODEL_CONFIG['name']}")
print(f"METHOD: Retain-Only LoRA Fine-Tuning ({NUM_EPOCHS} epochs, lr={LEARNING_RATE})")

forget_preds = collect_predictions(model, tokenizer, forget_set, "forget")
forget_results, overall_fsr = compute_metrics(forget_preds, "forget")

retain_preds = collect_predictions(model, tokenizer, retain_set, "retain")
retain_results, overall_rr = compute_metrics(retain_preds, "retain")

# Summary
categories = sorted(set(list(forget_results.keys()) + list(retain_results.keys())))
print(f"\n{'='*50}")
print(f"SUMMARY: Retain-Only LoRA Fine-Tuning")
print(f"Model: {MODEL_CONFIG['name']}  |  Epochs: {NUM_EPOCHS}  |  LR: {LEARNING_RATE}")
print(f"{'='*50}")
print(f"\n{'Category':<12} {'FSR (↑)':>10} {'RR (↑)':>10}")
print(f"{'-'*34}")
for cat in categories:
    fr = forget_results.get(cat, {"correct": 0, "total": 1})
    rr = retain_results.get(cat, {"correct": 0, "total": 1})
    fsr = 1.0 - (fr["correct"] / fr["total"]) if fr["total"] > 0 else 0
    rr_val = rr["correct"] / rr["total"] if rr["total"] > 0 else 0
    print(f"{cat:<12} {fsr:>10.1%} {rr_val:>10.1%}")
print(f"{'-'*34}")
print(f"{'OVERALL':<12} {overall_fsr:>10.1%} {overall_rr:>10.1%}")
print(f"{'='*34}")

## Upload predictions to HuggingFace

In [ ]:
# ── Upload predictions to HuggingFace ──────────────────────
from huggingface_hub import HfApi
import os

HF_REPO = "Novaspree/retain-only-finetuning-results"  # ← EDIT THIS

os.makedirs("./predictions", exist_ok=True)

model_tag = MODEL_CONFIG["name"].lower().replace(" ", "_").replace("-", "_")
forget_path = f"./predictions/forget_preds_{model_tag}_retain_ft.json"
retain_path = f"./predictions/retain_preds_{model_tag}_retain_ft.json"
all_path = f"./predictions/all_preds_{model_tag}_retain_ft.json"

all_preds = forget_preds + retain_preds

with open(forget_path, "w") as f:
    json.dump(forget_preds, f, indent=2)
with open(retain_path, "w") as f:
    json.dump(retain_preds, f, indent=2)
with open(all_path, "w") as f:
    json.dump(all_preds, f, indent=2)

print(f"Saved locally: {len(forget_preds)} forget + {len(retain_preds)} retain")

api = HfApi()
api.create_repo(HF_REPO, repo_type="dataset", exist_ok=True)

for fpath in [forget_path, retain_path, all_path]:
    api.upload_file(
        path_or_fileobj=fpath,
        path_in_repo=os.path.basename(fpath),
        repo_id=HF_REPO,
        repo_type="dataset",
    )

print(f"\n✓ Uploaded to https://huggingface.co/datasets/{HF_REPO}")

In [ ]:
# ── Save model (optional) ──────────────────────────────────
print(f"Saving unlearned model to: {OUTPUT_DIR}")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("✓ Model saved.")